# Model Training & Evaluation
This notebook demonstrates the training pipeline, hyperparameter tuning, and performance evaluation (ROC-AUC, Precision, Recall).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from ml.data.joiner import DataJoiner
from ml.feature_engineering.pipeline import FeatureEngineer
from ml.preprocessing.pipeline import Preprocessor
from ml.training.trainer import ModelTrainer
from ml.evaluation.metrics import Evaluator
from ml.evaluation.plots import Plotter

%matplotlib inline

In [ ]:
# 1. Load Data
df = DataJoiner().create_master_dataset()
df_features = FeatureEngineer().fit_transform(df)

y = df_features['risk_label']
X = df_features.drop(columns=['risk_label', 'transaction_id'], errors='ignore')

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
# 2. Preprocess
preprocessor = Preprocessor()
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)
X_train_processed.head(3)

In [ ]:
# 3. Train Models
trainer = ModelTrainer(random_state=42)
X_t, X_v, y_t, y_v = train_test_split(X_train_processed, y_train, test_size=0.1, random_state=42, stratify=y_train)

results, best_name, best_model = trainer.train_and_evaluate(X_t, y_t, X_v, y_v)
print(f"\nBest Model before tuning: {best_name}")

In [ ]:
# 4. Evaluate on Test Set
preds = best_model.predict(X_test_processed)
probs = best_model.predict_proba(X_test_processed)[:, 1]

metrics = Evaluator.get_metrics(y_test, preds, probs)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# 5. Plot ROC Curve
from IPython.display import Image
Plotter.plot_roc_curve(y_test, probs, model_name=best_name)
Image(filename='../../ml/evaluation/plots/roc_curve.png')